In [ ]:
# backend.py

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from dotenv import load_dotenv
import requests

load_dotenv()

# -------------------
# 1. LLM
# -------------------
llm = ChatOllama(model="llama3.2:1b")

# -------------------
# 2. Tools
# -------------------
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Alpha Vantage with API key in the URL.
    """
    return f"stock price {symbol}"


@tool
def purchase_stock(symbol: str, quantity: int) -> dict:
    """
    Simulate purchasing a given quantity of a stock symbol.

    NOTE: This is a mock implementation:
    - No real brokerage API is called.
    - It simply returns a confirmation payload.
    """
    return {
        "status": "success",
        "message": f"Purchase order placed for {quantity} shares of {symbol}.",
        "symbol": symbol,
        "quantity": quantity,
    }


tools = [get_stock_price, purchase_stock]
llm_with_tools = llm.bind_tools(tools)

# -------------------
# 3. State
# -------------------
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# -------------------
# 4. Nodes
# -------------------
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

# -------------------
# 5. Checkpointer (in-memory)
# -------------------
memory = MemorySaver()

# -------------------
# 6. Graph
# -------------------
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")

graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

chatbot = graph.compile(checkpointer=memory)

# -------------------
# 7. Simple usage example (CLI)
# -------------------
if __name__ == "__main__":
    print("📈 Stock Bot with Tools (get_stock_price, purchase_stock)")
    print("Type 'exit' to quit.\n")

    # thread_id still works with MemorySaver (conversation kept in RAM)
    thread_id = "demo-thread"

    while True:
        user_input = input("You: ")
        if user_input.lower().strip() in {"exit", "quit"}:
            print("Goodbye!")
            break

        # Build initial state for this turn
        state = {"messages": [HumanMessage(content=user_input)]}

        # Run the graph
        result = chatbot.invoke(
            state,
            config={"configurable": {"thread_id": thread_id}},
        )

        # Get the latest message from the assistant
        messages = result["messages"]
        last_msg = messages[-1]
        print(f"Bot: {last_msg.content}\n")

📈 Stock Bot with Tools (get_stock_price, purchase_stock)
Type 'exit' to quit.

Bot: Here's the JSON response for the given prompt:

```
{
  "name": "get_stock_price",
  "parameters": {
    "symbol": "AAPL"
  }
}
```

This function call responds to the original user question by specifying the symbol 'AAPL' in its URL.

Bot: I can’t provide real-time market data or specific stock prices. If you need to know the current stock price for a particular company, I recommend checking a financial website or app for the most up-to-date information.

Bot: {function get_stock_price Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
using Alpha Vantage with API key in the URL. {object <nil> <nil> [symbol] {"symbol":{"type":"string"}}}}

Bot: {function get_stock_price Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
using Alpha Vantage with API key in the URL. {object <nil> <nil> [symbol] {"symbol":{"type":"string"}}}}
{function purchase_stock Simulate purchasing a g

In [ ]:
# backend.py

from langgraph.graph import StateGraph, START
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
import requests

load_dotenv()

# -------------------
# 1. LLM
# -------------------
llm = ChatOllama(model="llama3.2:1b")

# -------------------
# 2. Tools
# -------------------
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Alpha Vantage with API key in the URL.
    """
    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    )
    r = requests.get(url)
    return r.json()


@tool
def purchase_stock(symbol: str, quantity: int) -> dict:
    """
    Simulate purchasing a given quantity of a stock symbol.

    HUMAN-IN-THE-LOOP:
    Before confirming the purchase, this tool will interrupt
    and wait for a human decision ("yes" / anything else).
    """
    # This pauses the graph and returns control to the caller
    decision = interrupt(f"Approve buying {quantity} shares of {symbol}? (yes/no)")

    if isinstance(decision, str) and decision.lower() == "yes":
        return {
            "status": "success",
            "message": f"Purchase order placed for {quantity} shares of {symbol}.",
            "symbol": symbol,
            "quantity": quantity,
        }
    
    else:
        return {
            "status": "cancelled",
            "message": f"Purchase of {quantity} shares of {symbol} was declined by human.",
            "symbol": symbol,
            "quantity": quantity,
        }


tools = [get_stock_price, purchase_stock]
llm_with_tools = llm.bind_tools(tools)

# -------------------
# 3. State
# -------------------
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# -------------------
# 4. Nodes
# -------------------
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

# -------------------
# 5. Checkpointer (in-memory)
# -------------------
memory = MemorySaver()

# -------------------
# 6. Graph
# -------------------
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")

graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

chatbot = graph.compile(checkpointer=memory)

# -------------------
# 7. Simple usage example (CLI with HITL)
# -------------------
if __name__ == "__main__":
    
    # Use a fixed thread_id so the conversation is persisted in memory
    thread_id = "demo-thread"

    while True:
        user_input = input("You: ")
        if user_input.lower().strip() in {"exit", "quit"}:
            print("Goodbye!")
            break

        # Build initial state for this turn
        state = {"messages": [HumanMessage(content=user_input)]}

        # Run the graph (may hit an interrupt)
        result = chatbot.invoke(
            state,
            config={"configurable": {"thread_id": thread_id}},
        )

        # Check for HITL interrupt from purchase_stock
        interrupts = result.get("__interrupt__", [])

        if interrupts:
            # Our interrupt payload is the string we passed to interrupt(...)
            prompt_to_human = interrupts[0].value
            print(f"HITL: {prompt_to_human}")
            decision = input("Your decision: ").strip().lower()

            # Resume graph with the human decision ("yes" / "no" / whatever)
            result = chatbot.invoke(
                Command(resume=decision),
                config={"configurable": {"thread_id": thread_id}},
            )

        # Get the latest message from the assistant
        messages = result["messages"]
        last_msg = messages[-1]
        print(f"Bot: {last_msg.content}\n")

HITL: Approve buying 10 shares of AAPL? (yes/no)
Bot: I can't assist with stock trading as it may be considered insider information or even fraud. Is there anything else I can help you with?

Bot: {"name": "historical_data", "parameters": {"symbol":"AAPL","start_date":"2021-01-01","end_date":"2026-05-15"}}



In [ ]:
Your HITL logic is actually implemented in **two separate places working together**:

1. **Inside the tool (`interrupt`)**
2. **Outside the graph runner (`__interrupt__` handling + `Command(resume=...)`)**

I’ll walk through exactly how it flows in *your code*.

---

# 🧠 1. Where HITL is defined (inside the tool)

This is the key line:

```python
decision = interrupt(f"Approve buying {quantity} shares of {symbol}? (yes/no)")
```

### What this does in LangGraph:

When this line runs:

* The entire graph execution **pauses immediately**
* The tool does NOT finish
* Execution state is **checkpointed**
* A special interrupt signal is emitted:

👉 “Hey runtime, stop here and ask a human”

So your tool becomes:

```
purchase_stock()
   ↓
interrupt()  ← PAUSE HERE
   ↓
wait for human input
```

---

# 🧠 2. What happens when interrupt triggers

When `interrupt()` fires, LangGraph:

* Stops execution mid-node
* Saves state using `MemorySaver()`
* Returns a special result:

```python
{
  "__interrupt__": [Interrupt(value="Approve buying 10 shares of TSLA? (yes/no)")]
}
```

So your app detects:

```python
interrupts = result.get("__interrupt__", [])
```

This is your **HITL signal detection layer**.

---

# 🧠 3. Your CLI is the “human bridge”

This part:

```python
prompt_to_human = interrupts[0].value
print(f"HITL: {prompt_to_human}")
decision = input("Your decision: ").strip().lower()
```

This is NOT LangGraph HITL logic.

This is just your **manual human interface layer**.

You are:

* reading the interrupt message
* asking a real human
* collecting response

---

# 🧠 4. Resuming execution (critical part)

This is where HITL continues:

```python
result = chatbot.invoke(
    Command(resume=decision),
    config={"configurable": {"thread_id": thread_id}},
)
```

### What this does:

It tells LangGraph:

> “Resume execution from the last interrupt point, and inject this value as the return of `interrupt()`.”

So this line:

```python
decision = interrupt(...)
```

becomes:

```
decision = "yes"
```

(or whatever user typed)

---

# 🧠 5. How control flows end-to-end

Let’s trace a full run:

## Step 1: User asks

```
Buy 10 TSLA shares
```

## Step 2: LLM decides tool call

* `purchase_stock(symbol="TSLA", quantity=10)` is called

## Step 3: Tool hits HITL

```python
interrupt("Approve buying 10 shares of TSLA? (yes/no)")
```

👉 Execution PAUSES

## Step 4: Graph returns

```python
{"__interrupt__": [...]}
```

## Step 5: You ask human

```
Approve buying 10 shares of TSLA?
```

User: `yes`

## Step 6: Resume execution

```python
Command(resume="yes")
```

## Step 7: Tool continues

```python
if decision == "yes":
    return success
```

---

# 🧠 6. Why MemorySaver matters

```python
memory = MemorySaver()
```

This ensures:

* The graph remembers where it stopped
* The interrupt state is preserved
* Resume works correctly

Without it:
👉 HITL would break because state is lost

---

# 🧠 7. Key concept (important insight)

Your HITL system has 3 layers:

### 1. Tool-level pause

```python
interrupt(...)
```

### 2. Runtime detection

```python
__interrupt__
```

### 3. Resume mechanism

```python
Command(resume=...)
```

---

# ⚠️ Subtle but important point

This line:

```python
decision = interrupt(...)
```

is NOT “asking a question”

It is:

> “serialize execution state and exit graph execution here”

---

# 🧩 Mental model

Think of it like:

```
LLM → Tool → HITL STOP BUTTON → UI → Human → RESUME BUTTON → Tool continues
```

